# Black Carbon Predictive Modeling

Train and evaluate regressors (Gradient Boosting, GRU) on merged dashcam + BC + traffic features.

**Sections** (run setup first, then any experiment block):
1. Setup & feature groups
2. Cross-validation: vehicles vs. GRU
3. Feature scenarios (lags, weather, traffic)
4. Sampling-interval comparison (1s–30s)
5. Holdout evaluation (20% tail)
6. Cross-collection-day transfer
7. Combined-day training
8. Personalization learning curve
9. SHAP interpretability

Core logic lives in `bc_modeling.py` (importable and unit-tested). See [README.md](README.md) for pipeline integration.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

from pathlib import Path

import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

from bc_config import (
    DEFAULT_BC_THRESHOLD,
    DEFAULT_EXCLUDE_FEATURES,
    DEFAULT_WEATHER_COLS,
)
from bc_modeling import (
    classify_feature_columns,
    compare_sampling_intervals,
    default_feature_scenarios,
    evaluate_algorithms,
    evaluate_feature_scenarios,
    evaluate_time_series_cross_val,
    improved_time_based_split,
    prepare_data,
    report_and_test,
    run_personalization_curve,
    seed_everything,
    export_shap_outputs,
    plot_vehicle_prediction_correlations,
)

PACKAGE_DIR = Path(".").resolve()
DATA_DIR = PACKAGE_DIR / "data"
OUTPUT_DIR = PACKAGE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(42)


## Setup — data paths & feature groups

Set CSV paths below to your merged `model_features` tables (see [README.md](README.md)).


In [ ]:
exclude_features = DEFAULT_EXCLUDE_FEATURES
threshold = DEFAULT_BC_THRESHOLD

PRIMARY_CSV = DATA_DIR / "model_features_5s_day2.csv"
TRANSFER_CSV = DATA_DIR / "model_features_5s.csv"

for label, path in [("Primary", PRIMARY_CSV), ("Transfer", TRANSFER_CSV)]:
    if not path.exists():
        raise FileNotFoundError(f"{label} input not found: {path}. See README.md.")

X, y = prepare_data(PRIMARY_CSV, exclude_features)
X_train_full, X_test_full, y_train_full, y_test_full = improved_time_based_split(
    X, y, train_ratio=0.8, n_splits=1
)
cv_splits = improved_time_based_split(X_train_full, y_train_full, n_splits=5)

groups = classify_feature_columns(X)
vehicle_cols = groups["vehicle_cols"]
traffic_cols = groups["traffic_cols"]
lag_10m = groups["lag_10m"]

best_features = vehicle_cols + DEFAULT_WEATHER_COLS + traffic_cols + lag_10m
best_features_ts = vehicle_cols + DEFAULT_WEATHER_COLS + traffic_cols

print(f"Rows: {len(X):,} | Train: {len(X_train_full):,} | Test holdout: {len(X_test_full):,}")
print(f"Features (full): {len(best_features)} | GRU subset: {len(best_features_ts)}")


## 2. Cross-validation — vehicle-only Gradient Boosting vs. GRU

In [ ]:
seed_everything(42)
gb_results = evaluate_algorithms(X[vehicle_cols], y, cv_splits, threshold, test_metric="MAE")
try:
    dl_results = evaluate_time_series_cross_val(
        X[vehicle_cols], y, cv_splits, threshold, test_metric="MAE", window_size=30
    )
    report_and_test({**gb_results, **dl_results}, test_metric="MAE", index_name="Model")
except ImportError as exc:
    print("Skipping GRU:", exc)


## 3. Feature scenarios

In [ ]:
seed_everything(42)
scenarios = default_feature_scenarios(groups)
scenario_results = evaluate_feature_scenarios(
    X, y, cv_splits, GradientBoostingRegressor(random_state=42), scenarios, threshold
)
try:
    evaluate_time_series_cross_val(
        X[best_features], y, cv_splits, threshold, test_metric="MAE", window_size=30
    )
except ImportError as exc:
    print("Skipping GRU on best features:", exc)


## 4a. Sampling intervals — Gradient Boosting

In [ ]:
INTERVAL_CSV_PATHS = {
    "1s": DATA_DIR / "model_features_1s_day2.csv",
    "5s": DATA_DIR / "model_features_5s_day2.csv",
    "10s": DATA_DIR / "model_features_10s_day2.csv",
    "15s": DATA_DIR / "model_features_15s_day2.csv",
    "30s": DATA_DIR / "model_features_30s_day2.csv",
}
interval_paths = {k: p for k, p in INTERVAL_CSV_PATHS.items() if p.exists()}
if not interval_paths:
    raise FileNotFoundError(
        "No interval CSVs found. Set INTERVAL_CSV_PATHS in this cell (see README.md)."
    )
feature_cols = vehicle_cols + DEFAULT_WEATHER_COLS + traffic_cols + lag_10m

compare_sampling_intervals(
    interval_paths,
    exclude_features,
    feature_cols,
    threshold,
    evaluator=evaluate_algorithms,
)


## 4b. Sampling intervals — GRU

In [ ]:
try:
    compare_sampling_intervals(
        interval_paths,
        exclude_features,
        feature_cols,
        threshold,
        evaluator=evaluate_time_series_cross_val,
        evaluator_kwargs={"window_size": 10},
    )
except ImportError as exc:
    print("Skipping interval GRU comparison:", exc)


## 5. Holdout evaluation (80/20 split on primary file)

In [ ]:
seed_everything(42)
holdout = [(X_train_full.index, X_test_full.index)]

evaluate_algorithms(X[best_features], y, holdout, threshold)
try:
    evaluate_time_series_cross_val(X[best_features_ts], y, holdout, threshold, window_size=10)
except ImportError as exc:
    print("Skipping holdout GRU:", exc)


## 6. Cross-day transfer (train primary → test transfer CSV)

Uses the 80% train split from the primary (Day 2) file and evaluates on the full transfer file.


In [ ]:
seed_everything(42)
X_transfer, y_transfer = prepare_data(TRANSFER_CSV, exclude_features)

model = GradientBoostingRegressor(random_state=42)
model.fit(X_train_full[best_features], y_train_full)
preds = model.predict(X_transfer[best_features])
from bc_modeling import calculate_metrics
print(pd.DataFrame([calculate_metrics(y_transfer, preds, threshold)], index=["Gradient Boosting"]))


## 7. Combined collections (70/30 split)

In [ ]:
df_a = pd.read_csv(PRIMARY_CSV)
df_b = pd.read_csv(TRANSFER_CSV)
combined = pd.concat([df_a, df_b], ignore_index=True)
from bc_eda import ensure_bc_cleaned
combined = ensure_bc_cleaned(combined.dropna())

X_comb = combined[best_features]
y_comb = combined["bc_cleaned"]
X_tr, X_te, y_tr, y_te = improved_time_based_split(X_comb, y_comb, train_ratio=0.7, block_size=10, n_splits=1)

model = GradientBoostingRegressor(random_state=42)
model.fit(X_tr, y_tr)
preds = model.predict(X_te)
print(calculate_metrics(y_te, preds, threshold))


## 8. Personalization learning curve

Fixed 20% test pool from transfer file; increasing fractions of the 80% pool added to primary-day training.


In [ ]:
seed_everything(42)
X_d2, y_d2 = prepare_data(TRANSFER_CSV, exclude_features)
X_d2 = X_d2[best_features]
X_pool, X_fixed_test, y_pool, y_fixed_test = improved_time_based_split(
    X_d2, y_d2, train_ratio=0.8, block_size=10, n_splits=1
)

X_d1, y_d1 = prepare_data(PRIMARY_CSV, exclude_features)
X_d1 = X_d1[best_features]
X_train_d1, _, y_train_d1, _ = improved_time_based_split(
    X_d1, y_d1, train_ratio=0.8, block_size=10, n_splits=1
)

curve = run_personalization_curve(
    X_train_d1, y_train_d1, X_pool, y_pool, X_fixed_test, y_fixed_test, threshold
)
print(curve[["Total_Day2_Percent", "MAE", "R2", "AUC", "Sensitivity", "Specificity"]].to_string(index=False))


## 9. SHAP & vehicle correlation plot

In [ ]:
# Re-fit best personalization model (100% pool) for explainability
slice_idx = len(X_pool)
X_train_combined = pd.concat([X_train_d1, X_pool.iloc[:slice_idx]], ignore_index=True)
y_train_combined = pd.concat([y_train_d1, y_pool.iloc[:slice_idx]], ignore_index=True)
explain_model = GradientBoostingRegressor(random_state=42)
explain_model.fit(X_train_combined, y_train_combined)
preds = explain_model.predict(X_fixed_test)

shap_dir = OUTPUT_DIR / "shap"
vehicle_df, summary_df = export_shap_outputs(
    explain_model, X_fixed_test, preds, output_dir=shap_dir
)
plot_vehicle_prediction_correlations(
    shap_dir / "predicted_black_carbon_results.csv",
    OUTPUT_DIR / "updated_vehicle_impact.png",
)
print("SHAP exports written to", shap_dir)
